# 5. Combining Profiles

## Purpose
Concatenate all per-well-FOV parquet files for a single patient into three
patient-level combined parquets (SC, organoid, nucleocentric).

This is **step 5 of Stage 4 (image-based profiling)**. It runs once per patient
and is typically submitted as a per-patient SLURM job.

## Inputs
- `data/{patient}/image_based_profiles/1.related_profiles/{well_fov}/`
  - `sc_profiles_{well_fov}_related.parquet`
  - `organoid_profiles_{well_fov}_related.parquet`
  - `nucleocentric_profiles_{well_fov}_related.parquet`

## Outputs
Three combined parquets in `data/{patient}/image_based_profiles/2.combined_profiles/`:

| File | Content |
|---|---|
| `sc.parquet` | All SC profiles stacked across FOVs |
| `organoid.parquet` | All organoid profiles stacked across FOVs |
| `nucleocentric.parquet` | All nucleocentric profiles stacked across FOVs |

## Notes
- Concatenation uses DuckDB `union_by_name=true`, which aligns columns by name
  rather than position. FOVs with missing columns (e.g. empty scaffold tables)
  will have those columns filled with NULL.
- Brightfield (BF) channel features are removed after concatenation as they are
  not part of the fluorescent cell painting panel and are not used in profiling.

In [1]:
import os
import pathlib

import duckdb
import pandas as pd
from image_analysis_3D.file_utils.arg_parsing_utils import parse_args
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)
profile_base_dir = root_dir

In [2]:
if not in_notebook:
    args = parse_args()
    patient = args["patient"]
    image_based_profiles_subparent_name = args["image_based_profiles_subparent_name"]

else:
    patient = "NF0037_T1_CQ1"
    image_based_profiles_subparent_name = "image_based_profiles"

In [3]:
# set paths
profiles_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/1.related_profiles"
).resolve(strict=True)
# output_paths
sc_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/sc.parquet"
).resolve()
organoid_merged_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/organoid.parquet"
).resolve()
nucleocentric_profile_output_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{image_based_profiles_subparent_name}/2.combined_profiles/nucleocentric.parquet"
).resolve()
organoid_merged_output_path.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Discover all per-FOV parquet files under 1.related_profiles/.
# The directory structure is 1.related_profiles/{well_fov}/*.parquet,
# so one wildcard level is sufficient.
profiles = list(profiles_path.rglob("*/*.parquet"))

In [5]:
# Split files by profile type using filename prefix.
# Expected prefixes: 'sc_', 'organoid_', 'nucleocentric_'.
sc_profiles = [str(x) for x in profiles if x.name.startswith("sc_")]
organoid_profiles = [str(x) for x in profiles if x.name.startswith("organoid_")]
nucleocentric_profiles = [
    str(x) for x in profiles if x.name.startswith("nucleocentric_")
]

In [6]:
for x in nucleocentric_profiles:
    df = pd.read_parquet(x)
    if df.isnull().any().any():
        print(f"Null values found in {x}")
df

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,1,E5-6,-0.195807,-0.274601,0.213762,-0.021315,-0.152792,0.223191,0.014228,-0.119720,...,3.484517,-5.384260,4.388068,4.055420,4.461672,0.428795,2.874498,0.861912,-1.236112,-1
1,4,E5-6,-0.250884,-0.301353,0.220031,-0.009122,-0.155448,0.211776,0.028734,-0.110621,...,4.341091,-7.318145,-3.180063,1.991689,-3.774339,4.286053,1.143398,-0.323755,-0.179799,-1
2,6,E5-6,-0.312636,-0.323805,0.244319,-0.007859,-0.175337,0.147521,0.033872,-0.058905,...,4.405370,-2.320999,3.834863,3.473490,-5.341408,2.816374,4.151417,-1.782873,-3.430293,-1
3,7,E5-6,-0.036723,-0.082613,0.058807,-0.202426,0.091331,0.248103,0.031806,-0.080753,...,5.545275,-9.030755,2.846426,2.441857,2.242561,2.244371,6.020228,2.231839,0.767033,-1
4,8,E5-6,-0.208556,-0.243058,0.191097,-0.044324,-0.070343,0.212904,0.054547,-0.097996,...,3.379646,-8.260204,0.055813,-0.253129,1.938643,-0.370153,4.829574,4.132085,0.297872,-1


In [7]:
# Concatenate per-FOV parquets for each profile type using DuckDB.
# union_by_name=true aligns columns by name rather than position, so FOVs with
# differing column sets (e.g. empty scaffold tables from notebook 1) are handled
# gracefully — missing columns are filled with NULL rather than causing an error.

with duckdb.connect() as conn:
    sc_profile = conn.execute(
        f"SELECT * FROM read_parquet({sc_profiles}, union_by_name=true)"
    ).df()
    organoid_profile = conn.execute(
        f"SELECT * FROM read_parquet({organoid_profiles}, union_by_name=true)"
    ).df()
    nucleocentric_profile = conn.execute(
        f"SELECT * FROM read_parquet({nucleocentric_profiles}, union_by_name=true)"
    ).df()

print(f"Single-cell profiles concatenated. Shape: {sc_profile.shape}")
print(f"Organoid profiles concatenated. Shape: {organoid_profile.shape}")
print(f"Nucleocentric profiles concatenated. Shape: {nucleocentric_profile.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Single-cell profiles concatenated. Shape: (7432, 11889)
Organoid profiles concatenated. Shape: (1085, 3962)
Nucleocentric profiles concatenated. Shape: (7432, 3075)


## Remove all BF channels


In [8]:
# Remove brightfield (BF) channel features from all three profile types.
# BF is a transmitted-light channel not part of the fluorescent cell painting
# panel; its features are may not meaningful for morphological profiling.
# and are interpreted differently
# Note: if no BF columns exist in the data, these drops are no-ops.

bf_cols_sc = [col for col in sc_profile.columns if "BF" in col]
sc_profile = sc_profile.drop(columns=bf_cols_sc)
print(f"SC: dropped {len(bf_cols_sc)} BF columns. Shape: {sc_profile.shape}")

bf_cols_organoid = [col for col in organoid_profile.columns if "BF" in col]
organoid_profile = organoid_profile.drop(columns=bf_cols_organoid)
print(
    f"Organoid: dropped {len(bf_cols_organoid)} BF columns. Shape: {organoid_profile.shape}"
)

bf_cols_nucleocentric = [col for col in nucleocentric_profile.columns if "BF" in col]
nucleocentric_profile = nucleocentric_profile.drop(columns=bf_cols_nucleocentric)
print(
    f"Nucleocentric: dropped {len(bf_cols_nucleocentric)} BF columns. Shape: {nucleocentric_profile.shape}"
)

SC: dropped 0 BF columns. Shape: (7432, 11889)
Organoid: dropped 0 BF columns. Shape: (1085, 3962)
Nucleocentric: dropped 0 BF columns. Shape: (7432, 3075)


In [9]:
sc_profile.to_parquet(sc_merged_output_path, index=False)
organoid_profile.to_parquet(organoid_merged_output_path, index=False)
nucleocentric_profile.to_parquet(nucleocentric_profile_output_path, index=False)

In [10]:
nucleocentric_profile

,object_id,image_set,Nucleocentric_Mito_SAMMed3D_Feature0,Nucleocentric_Mito_SAMMed3D_Feature1,Nucleocentric_Mito_SAMMed3D_Feature10,Nucleocentric_Mito_SAMMed3D_Feature100,Nucleocentric_Mito_SAMMed3D_Feature101,Nucleocentric_Mito_SAMMed3D_Feature102,Nucleocentric_Mito_SAMMed3D_Feature103,Nucleocentric_Mito_SAMMed3D_Feature104,...,Nucleocentric_ER_CHAMMI75_Feature91,Nucleocentric_ER_CHAMMI75_Feature92,Nucleocentric_ER_CHAMMI75_Feature93,Nucleocentric_ER_CHAMMI75_Feature94,Nucleocentric_ER_CHAMMI75_Feature95,Nucleocentric_ER_CHAMMI75_Feature96,Nucleocentric_ER_CHAMMI75_Feature97,Nucleocentric_ER_CHAMMI75_Feature98,Nucleocentric_ER_CHAMMI75_Feature99,ParentOrganoid
0,1,D6-8,-0.407552,-0.301788,0.266882,-0.008465,-0.183757,0.099491,0.006847,-0.061775,...,2.833289,-7.215891,2.171409,5.539629,-0.159016,-1.310103,4.169992,1.456856,-0.383529,-1
1,2,D6-8,-0.471192,-0.329269,0.262076,-0.008872,-0.197235,0.107374,-0.000628,-0.046665,...,3.047008,-8.746221,0.704551,-0.983699,-1.531238,-4.464877,2.821905,0.902721,-1.903956,1
2,3,D6-8,-0.434409,-0.318036,0.237594,0.075187,-0.197595,0.147974,-0.035164,-0.023854,...,3.672410,-4.712862,3.192608,1.989115,-0.640443,-0.565081,3.996702,1.482946,-1.127162,-1
3,4,D6-8,-0.360831,-0.280258,0.228794,-0.045785,-0.171328,0.168033,0.040686,-0.104048,...,4.398346,-8.736972,0.615667,1.666963,-2.765219,-2.455949,4.480609,2.508102,-0.014121,-1
4,5,D6-8,-0.147715,-0.246267,0.268715,-0.028098,-0.161025,0.137257,0.070864,-0.113034,...,4.980976,-7.185730,2.696625,0.671308,-1.108319,-4.526272,2.763840,0.539867,-1.782517,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7427,1,E5-6,-0.195807,-0.274601,0.213762,-0.021315,-0.152792,0.223191,0.014228,-0.119720,...,3.484517,-5.384260,4.388068,4.055420,4.461672,0.428795,2.874498,0.861912,-1.236112,-1
7428,4,E5-6,-0.250884,-0.301353,0.220031,-0.009122,-0.155448,0.211776,0.028734,-0.110621,...,4.341091,-7.318145,-3.180063,1.991689,-3.774339,4.286053,1.143398,-0.323755,-0.179799,-1
7429,6,E5-6,-0.312636,-0.323805,0.244319,-0.007859,-0.175337,0.147521,0.033872,-0.058905,...,4.405370,-2.320999,3.834863,3.473490,-5.341408,2.816374,4.151417,-1.782873,-3.430293,-1
7430,7,E5-6,-0.036723,-0.082613,0.058807,-0.202426,0.091331,0.248103,0.031806,-0.080753,...,5.545275,-9.030755,2.846426,2.441857,2.242561,2.244371,6.020228,2.231839,0.767033,-1
